# Introduction

Cart abandonment is a critical challenge for digital ordering platforms, directly impacting revenue and customer retention. For MyCoke360, Coca-Cola's B2B digital ordering system launched in Summer 2024, understanding why customers fail to complete purchases is especially important. The platform serves Food Service On Premise (FSOP) customers such as restaurants, schools, hospitals, and retailers, where order frequency and product mix drive significant business value. By examining customer behavior captured in Google Analytics alongside order and sales data, this project seeks to uncover patterns that explain when, how, and why carts are abandoned.

This exploratory data analysis (EDA) will focus on evaluating the quality, structure, and usability of the available data so that it is ready to be used for financial evaluation modeling in the later modeling stage. Other aspects of the problem statement such as identifying behavioral predictors, analyzing recovery patterns, and evaluating device-specific abandonment will be addressed by other members of the project team. This division of workflow ensures comprehensive coverage of the problem space while allowing each stage of the analysis to build on a solid data foundation.

## Initial Guiding Questions

- What is the most efficient method for representing the previously defined cart abondonment within the data?
- How can abandoned carts be aggregated to accurately estimate lost revenue at both the order and product level?
- Which product categories, pack types, or SKUs appear most frequently in abandoned carts, and how should these be visualized for clear insights?
- What is the distribution of abandonment across different customer segments, such as sales office, plant, or FSOP type?
- How can abandoned cart revenue be compared against total sales to highlight the relative financial impact?
- What temporal patterns emerge in abandoned carts (e.g., by day of week, order cycle, or over time during the study period)?
- Are there systematic differences in abandonment linked to operational factors such as cutoff times or anchor days?

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml import Pipeline

spark.conf.set("spark.sql.session.timeZone", "UTC")

# Data Loading & Structure

The dataset consists of eight CSV tables covering customer behavior, transactions, and supporting reference information. Three fact tables capture activity on MyCoke360: Google Analytics events (site visits, add/remove cart actions, purchases, and device/page details), Orders (materials ordered per customer, with order type and timestamps in both EST and UTC), and Sales (fulfilled transactions with pricing and profit measures). These are complemented by five dimension tables: Customer (account and channel attributes, sales office details), Cutoff Times (order cutoff policies by plant, office, and distribution mode), Material (product master data such as pack type, brand, flavor, and category), Operating Hours (current ordering frequency and anchor day/date by customer), and Visit Plan (historical anchor dates, frequencies, and sales office attributes). Collectively, these tables create a comprehensive view of both customer behavior and business processes adequetely enabling our analysis.

In [0]:
# Import fact tables
google_analytics = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/google_analytics.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)
orders = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/orders.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)
sales = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/sales.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)

# Import dimension tables
customers = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/customer.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)
cutoff_times = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/cutoff_times.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)
materials = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/material.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)
operating_hours = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/operating_hours.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)
visit_plan = spark.read.csv(
    "/Volumes/workspace/default/capstone_data/visit_plan.csv",
    header=True,
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True
)

# Initial Quality Checks

## Data Cleaning

## Data Types

In [0]:
display(google_analytics.limit(5))

items_schema = ArrayType(
    StructType([
        StructField("item_id", StringType()),
        StructField("quantity", StringType())
    ])
)

google_analytics = (
    google_analytics
    # .withColumn("EVENT_DATE", to_date("EVENT_DATE", "yyyy-MM-dd"))
    .withColumn("EVENT_TIMESTAMP", to_timestamp("EVENT_TIMESTAMP", "yyyy-MM-dd'T'HH:mm:ss.SSSX"))
    .withColumn(
        "ITEMS",
        from_json(col("ITEMS"), items_schema).cast("array<struct<item_id:string, quantity:int>>")
    )
    .select(
        "CUSTOMER_ID", 
        "EVENT_TIMESTAMP", 
        "EVENT_NAME", 
        "DEVICE_CATEGORY", 
        "DEVICE_MOBILE_BRAND_NAME", 
        "DEVICE_OPERATING_SYSTEM", 
        "EVENT_PAGE_NAME", 
        "EVENT_PAGE_TITLE", 
        "ITEMS"
    )
)

display(google_analytics.limit(5))
google_analytics.printSchema()

In [0]:
display(orders.limit(5))

orders = (
    orders
    .withColumn(
        "CREATED_DATE",
        to_timestamp("CREATED_DATE_UTC", "yyyy-MM-dd'T'HH:mm:ss.SSSX")
    )
    .withColumn(
        "ORDER_QUANTITY",
        regexp_replace("ORDER_QUANTITY", ",", "").cast("double")
    )
    .drop("CREATED_DATE_UTC", "CREATED_DATE_EST")
    .select(
        "CUSTOMER_ID",
        "MATERIAL_ID",
        "PLANT_ID",
        "ORDER_TYPE",
        "ORDER_QUANTITY",
        "CREATED_DATE"
    )
)

display(orders.limit(5))
orders.printSchema()

Observations:
- CREATED_DATE_EST is a date in EST, original format yyyy-MM-dd
- CREATED_DATE_UTC is a timestamp in UTC, original format yyyy-MM-ddTHH:mm:SS.SSSZ
- ORDER_QUANTITY is a float, but likely clear to be set as an integer as it is a count

In [0]:
display(sales.limit(5))

sales = (
    sales
    .withColumn("POSTING_DATE", to_date("POSTING_DATE", "M/d/yyyy"))
    .withColumn(
        "GROSS_PROFIT_DEAD_NET",
        regexp_replace("GROSS_PROFIT_DEAD_NET", ",", "").cast("double")
    )
    .withColumn(
        "PHYSICAL_VOLUME",
        regexp_replace("PHYSICAL_VOLUME", ",", "").cast("double")
    )
    .withColumn(
        "NSI_DEAD_NET",
        regexp_replace("NSI_DEAD_NET", ",", "").cast("double")
    )
    .select(
        "CUSTOMER_ID",
        "MATERIAL_ID",
        "POSTING_DATE",
        "GROSS_PROFIT_DEAD_NET",
        "NSI_DEAD_NET",
        "PHYSICAL_VOLUME"
    )
)

display(sales.limit(5))
sales.printSchema()

Observations:
- POSTING_DATE is a date, likely utc as it has no other identifier (check notes), original format M/dd/yyyy
- GROSS_PROFIT_DEAD_NET is a monetary float value, consider comma removal before casting
- PHYSICAL_VOLUME is a float value, consider comma removal before casting
- NSI_DEAD_NET is a float value, consider comma removal before casting

In [0]:
display(customers.limit(5))
customers.select("DISTRIBUTION_MODE_DESCRIPTION").distinct().show()

state_tz = {
    "AL":"America/Chicago",       "AK":"America/Anchorage",		"AZ":"America/Phoenix",
    "AR":"America/Chicago",       "CA":"America/Los_Angeles"	,"CO":"America/Denver",
    "CT":"America/New_York",      "DC":"America/New_York",		"DE":"America/New_York",
    "FL":"America/New_York",      "GA":"America/New_York",		"HI":"Pacific/Honolulu",
    "ID":"America/Denver",        "IL":"America/Chicago",		"IN":"America/Indiana/Indianapolis",
    "IA":"America/Chicago",       "KS":"America/Chicago",		"KY":"America/New_York",
    "LA":"America/Chicago",       "ME":"America/New_York",		"MD":"America/New_York",
    "MA":"America/New_York",      "MI":"America/Detroit",		"MN":"America/Chicago",
    "MS":"America/Chicago",       "MO":"America/Chicago",		"MT":"America/Denver",
    "NE":"America/Chicago",       "NV":"America/Los_Angeles",	"NH":"America/New_York",
    "NJ":"America/New_York",      "NM":"America/Denver",		"NY":"America/New_York",
    "NC":"America/New_York",      "ND":"America/Chicago",		"OH":"America/New_York",
    "OK":"America/Chicago",       "OR":"America/Los_Angeles",	"PA":"America/New_York",
    "RI":"America/New_York",      "SC":"America/New_York",		"SD":"America/Chicago",
    "TN":"America/Chicago",       "TX":"America/Chicago",		"UT":"America/Denver",
    "VT":"America/New_York",      "VA":"America/New_York",		"WA":"America/Los_Angeles",
    "WV":"America/New_York",      "WI":"America/Chicago",		"WY":"America/Denver"
}

customers = (
    customers
	.withColumn("STATE", regexp_extract(upper(col("SALES_OFFICE_DESCRIPTION")), r"([A-Z]{2})$", 1))
    .withColumn(
        "SALES_OFFICE_TZ",
        coalesce(
            create_map([lit(x) for kv in state_tz.items() for x in kv])[col("STATE")],
            lit(None)
        )
    )
    .select(
        "CUSTOMER_NUMBER",
        "SALES_OFFICE",
        col("SALES_OFFICE_DESCRIPTION").alias("SALES_OFFICE_DESC"),
        col("DISTRIBUTION_MODE_DESCRIPTION").alias("DISTRIBUTION_MODE_DESC"),
        col("SHIPPING_CONDITIONS_DESCRIPTION").alias("SHIPPING_CONDITIONS_DESC"),
        col("COLD_DRINK_CHANNEL_DESCRIPTION").alias("COLD_DRINK_CHANNEL_DESC"),
        col("CUSTOMER_SUB_TRADE_CHANNEL_DESCRIPTION").alias("CUSTOMER_SUB_TRADE_CHANNEL_DESC"),
        "STATE",
        "SALES_OFFICE_TZ"
    )
)

display(customers.limit(5))
customers.printSchema()

sales_office_tz = (
    customers
    .select(
        "SALES_OFFICE",
        "SHIPPING_CONDITIONS_DESC",
        "DISTRIBUTION_MODE_DESC",
        "SALES_OFFICE_TZ"
    )
    .distinct()
    .dropna(subset=["SHIPPING_CONDITIONS_DESC"])
)

# sales_office_tz.select("DISTRIBUTION_MODE").distinct().show()

display(sales_office_tz.limit(5))
sales_office_tz.printSchema()

Where the customer table is concerned, the only numerical column is CUSTOMER_NUMBER. Since this is used as an identifier the standard is to leave it as a string datatype. The rest of the columns within the dataset are either categorical or descriptions which remain strings as well.

In [0]:
display(visit_plan.limit(5))

visit_plan = (
	visit_plan
    .withColumn(
        "SHIPPING_CONDITION_TIME",
        when(col("SHIPPING_CONDITIONS_DESC").like("%24%"), lit("24hrs"))
        .when(col("SHIPPING_CONDITIONS_DESC").like("%48%"), lit("48hrs"))
        .when(col("SHIPPING_CONDITIONS_DESC").like("%72%"), lit("72hrs"))
        .otherwise(None)
    )
    # .withColumn("SNAPSHOT_DATE", to_date("SNAPSHOT_DATE", "yyyy-MM-dd"))
    # .withColumn("ANCHOR_DATE", to_date("ANCHOR_DATE", "yyyy-MM-dd"))
    .withColumn(
        "ELT_TS",
        to_timestamp("ELT_TS", "yyyy-MM-dd'T'HH:mm:ss.SSSX")
    )
)
display(visit_plan.limit(5))
visit_plan.printSchema()


# +------------------------+
# |SHIPPING_CONDITIONS_DESC|
# +------------------------+
# |                    NULL|
# |                    null|
# |                 24Hours|
# |                 72Hours|
# |         Dropsite48Hours|
# |         Dropsite24Hours|
# |         Dropsite72Hours|
# |                 48Hours|
# +------------------------+

visit_plan.select("SHIPPING_CONDITIONS_DESC").distinct().show()


In [0]:
display(cutoff_times.limit(5))

mode_map = {
    "OFS": "OF",
    "Rapid Delivery": "RD",
    "E Pallet": "EZ",
    "Sideload": "SL",
    "Night Sideload": "NS",
    "Full Service": "FS",
    "Night Rapid Delivery": "NR",
    "Night OFS": "NO",
    "Special Events": "SE",
    "Bulk Distribution": "BK"
}

cutoff_times.select("DISTRIBUTION_MODE").distinct().show()

cutoff_times = (
    cutoff_times
    .withColumn(
        "DISTRIBUTION_MODE",
        create_map([lit(x) for kv in mode_map.items() for x in kv])[col("DISTRIBUTION_MODE")]
    )
    # .join(
    #     sales_office_tz,
    #     on=["SALES_OFFICE", "SHIPPING_CONDITION_TIME", "DISTRIBUTION_MODE"], 
    #     how="left"
    # )
)

display(cutoff_times.limit(5))
cutoff_times.select("SHIPPING_CONDITION_TIME").distinct().show()


# sales_office_tz
#           "SALES_OFFICE",
#           "SHIPPING_CONDITIONS_DESC",
#           "DISTRIBUTION_MODE"

display(cutoff_times.limit(5))
cutoff_times.printSchema()

Observations:
- CUTOFFTIME__C is an AM/PM time, original format h:MM:SS AM
- SHIPPING_CONDITION_TIME looks to be a list of hours, may convert to integers (check first)

In [0]:
display(materials.limit(5))

# materials = (
#     materials
#     .withColumn()
# )

display(materials.limit(5))
materials.printSchema()

Observations:
- No data types need changing here, they all should be strings

In [0]:
display(operating_hours.limit(5))

operating_hours = (
    operating_hours
    # .withColumn()
)

display(operating_hours.limit(5))
operating_hours.printSchema()

Observations:
- FREQUENCY might be changed to integers (check first)
- DELIVERY_ANCHOR_DAY might be changed to integer week representations
- CALLING_ANCHOR_DATE is a date, original format d/MM/yyyy

Observations:
- FREQUENCY might be changed to an integer (check unique values first)
- ELT_TS is a utc timestamp, original format yyyy-MM-ddTHH:mm:SS.SSSZ
- SNAPSHOT_DATE is a date, original format yyyy-MM-dd
- ANCHOR_DATE is a date, original format yyyy-MM-dd

## Missing Values

In [0]:
def get_null_counts(df):
    null_counts = df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in df.columns
    ])

    transposed_null_counts = (
        null_counts
        .toPandas()
        .transpose()
        .reset_index()
    )
    transposed_null_counts.columns = ["column_name", "null_count"]

    display(transposed_null_counts)

In [0]:
get_null_counts(google_analytics)
display(google_analytics.limit(5))

google_analytics_orders = (
    google_analytics
    .filter(col("EVENT_NAME") == "purchase")
    .withColumn("ITEM", explode(col("ITEMS")))
    .withColumn("ITEM_ID", col("ITEM.item_id")) 
    .withColumn("ORDER_QUANTITY", col("ITEM.quantity"))
    .withColumn("ORDER_DATE", to_date(col("EVENT_TIMESTAMP")))
    .drop("ITEM", "ITEMS", "EVENT_PAGE_TITLE", "EVENT_NAME", "EVENT_TIMESTAMP")
)
display(google_analytics_orders)


In [0]:
get_null_counts(orders)
orders = (
    orders
    .withColumn("ORDER_DATE", to_date(col("CREATED_DATE")))
)
display(
    orders
    .filter(col("MATERIAL_ID").isNull())
    .join(
        google_analytics_orders,
        on=["CUSTOMER_ID", "ORDER_DATE", "ORDER_QUANTITY"],
        how="left"
    )
)

# orders:
# MATERIAL_ID	195
# PLANT_ID	11

In [0]:
get_null_counts(sales)


In [0]:
get_null_counts(customers)

# customers
# DISTRIBUTION_MODE_DESCRIPTION	4

In [0]:
get_null_counts(cutoff_times)


In [0]:
get_null_counts(materials)

# materials
# EV_CAT_DESC	134

In [0]:
get_null_counts(operating_hours)


In [0]:
get_null_counts(visit_plan)

# visit_plan
# DISTRIBUTION_MODE	15223
# SHIPPING_CONDITIONS_DESC	1

In [0]:
import matplotlib.pyplot as plt

# Convert just the column you need
df_pd = google_analytics.select("CUSTOMER_ID").toPandas().astype({'CUSTOMER_ID': 'float'})

# Then plot with pandas/matplotlib
df_pd["CUSTOMER_ID"].plot(kind="density")
plt.show()


## Duplicates

# Data Cleaning

# Univariate Analysis

## Outlier Detection

# Bivariate Analysis

# Multivariate Analysis

# Target Variable Analysis

# Feature Engineering

# Statistical Testing

# Summary of Findings